# NUS ST3131 — Regression Analysis
## Tutor-style Kaggle case study: Medical Insurance Charges

This notebook develops the main ideas of **ST3131 Regression Analysis** through one coherent case study rather than disconnected examples.

**Dataset:** Kaggle — *Medical Cost Personal Datasets* (`mirichoi0218/insurance`)

**Response**

\[
Y = \text{medical insurance charges}
\]

**Predictors**

- `age`
- `sex`
- `bmi`
- `children`
- `smoker`
- `region`

## Learning goals

You will implement and interpret:

1. exploratory analysis for regression,
2. simple linear regression,
3. multiple linear regression,
4. \(Y=X\beta+\epsilon\),
5. coefficient inference and confidence intervals,
6. overall and partial \(F\)-tests,
7. confidence intervals vs prediction intervals,
8. dummy variables,
9. one-way and two-way ANOVA,
10. ANCOVA,
11. interactions,
12. model building,
13. residual diagnostics,
14. leverage, studentized residuals and Cook's distance,
15. heteroscedasticity and robust standard errors,
16. multicollinearity and VIF,
17. transformations,
18. held-out predictive checking.

All visualizations use **Bokeh**.

## 0. How to use this notebook

Run from top to bottom.

The loader tries:

1. a local `insurance.csv`,
2. Kaggle via `kagglehub`,
3. a public mirror of the same source data as a fallback.

> **Tutor habit:** inspect the data and plots before looking at p-values. Statistical significance cannot rescue a poorly specified model.

In [72]:
# Uncomment if packages are missing:
#%pip install -q pandas numpy scipy statsmodels scikit-learn bokeh kagglehub

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from bokeh.io import output_notebook, show
from bokeh.layouts import row
from bokeh.models import ColumnDataSource, HoverTool, Span, NumeralTickFormatter
from bokeh.plotting import figure
from bokeh.transform import factor_cmap

warnings.filterwarnings("ignore")
output_notebook()

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

Loading BokehJS ...

## 1. Load the Kaggle dataset

This dataset is well suited to ST3131 because it mixes continuous, count and categorical predictors with a continuous response.

In [3]:
def load_insurance_data():
    candidates = [
        Path("insurance.csv"),
        Path("/kaggle/input/insurance/insurance.csv"),
        Path("/kaggle/input/medical-cost-personal-datasets/insurance.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate), f"local file: {candidate}"

    try:
        import kagglehub
        folder = Path(kagglehub.dataset_download("mirichoi0218/insurance"))
        matches = list(folder.rglob("insurance.csv"))
        if matches:
            return pd.read_csv(matches[0]), f"Kaggle via kagglehub: {matches[0]}"
    except Exception as exc:
        print("Kaggle download unavailable:", type(exc).__name__, str(exc)[:160])

    fallback_url = (
        "https://raw.githubusercontent.com/"
        "stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
    )
    return pd.read_csv(fallback_url), "public mirror of source data"

df, DATA_SOURCE = load_insurance_data()
print("Loaded from:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())

100%|█████████████████████████████████████████████████████████████████████████████| 16.0k/16.0k [00:00<00:00, 2.71MB/s]

Extracting files...
Loaded from: Kaggle via kagglehub: /home/anirban/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1/insurance.csv
Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.9000,0,yes,southwest,"16,884.9240"
1,18,male,33.7700,1,no,southeast,"1,725.5523"
2,28,male,33.0000,3,no,southeast,"4,449.4620"
3,33,male,22.7050,0,no,northwest,"21,984.4706"
4,32,male,28.8800,0,no,northwest,"3,866.8552"


In [4]:
display(df.describe(include="all").T)

print("Missing values")
display(df.isna().sum().to_frame("missing"))

print("Duplicate rows:", int(df.duplicated().sum()))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,"1,338.0000",NaN,NaN,NaN,39.2070,14.0500,18.0000,27.0000,39.0000,51.0000,64.0000
sex,1338,2,male,676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bmi,"1,338.0000",NaN,NaN,NaN,30.6634,6.0982,15.9600,26.2963,30.4000,34.6938,53.1300
children,"1,338.0000",NaN,NaN,NaN,1.0949,1.2055,0.0000,0.0000,1.0000,2.0000,5.0000
smoker,1338,2,no,1064,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,1338,4,southeast,364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
charges,"1,338.0000",NaN,NaN,NaN,"13,270.4223","12,110.0112","1,121.8739","4,740.2872","9,382.0330","16,639.9125","63,770.4280"


Missing values


,missing
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


Duplicate rows: 1


## 2. Understand the response

Before fitting

\[
Y_i=\beta_0+\beta_1x_i+\epsilon_i,
\]

inspect the shape of \(Y\).

Strong right-skew can later appear as non-normality or heteroscedasticity in residuals.

In [5]:
def bokeh_histogram(series, title, x_label, bins=30):
    values = pd.Series(series).dropna().to_numpy()
    hist, edges = np.histogram(values, bins=bins)

    p = figure(
        width=820, height=380,
        title=title,
        x_axis_label=x_label,
        y_axis_label="Count",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    p.quad(
        top=hist, bottom=0,
        left=edges[:-1], right=edges[1:],
        alpha=0.65
    )
    return p

show(
    bokeh_histogram(
        df["charges"],
        "Distribution of Medical Insurance Charges",
        "Charges",
        bins=35
    )
)

In [6]:
charge_summary = df["charges"].agg(["mean", "median", "std", "min", "max", "skew"])
display(charge_summary.to_frame("charges"))

,charges
mean,"13,270.4223"
median,"9,382.0330"
std,"12,110.0112"
min,"1,121.8739"
max,"63,770.4280"
skew,1.5159


## 3. Exploratory regression thinking

Ask:

- Does age show an approximately linear trend?
- Does smoking separate the population?
- Does BMI behave differently for smokers and non-smokers?

A good regression formula should represent visible structure rather than being chosen blindly.

In [7]:
smoker_factors = ["no", "yes"]
source = ColumnDataSource(df.copy())

p = figure(
    width=860, height=450,
    title="Charges vs Age, grouped by Smoking Status",
    x_axis_label="Age",
    y_axis_label="Charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    x="age", y="charges",
    source=source,
    size=7,
    alpha=0.50,
    color=factor_cmap(
        "smoker",
        palette=["#4C78A8", "#F58518"],
        factors=smoker_factors
    ),
    legend_field="smoker"
)

p.add_tools(HoverTool(tooltips=[
    ("age", "@age"),
    ("BMI", "@bmi{0.0}"),
    ("smoker", "@smoker"),
    ("charges", "@charges{$0,0.00}")
]))
p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
p.legend.title = "Smoker"
show(p)

In [8]:
p = figure(
    width=860, height=450,
    title="Charges vs BMI, grouped by Smoking Status",
    x_axis_label="BMI",
    y_axis_label="Charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    x="bmi", y="charges",
    source=source,
    size=7,
    alpha=0.50,
    color=factor_cmap(
        "smoker",
        palette=["#4C78A8", "#F58518"],
        factors=smoker_factors
    ),
    legend_field="smoker"
)

p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
p.legend.title = "Smoker"
show(p)

# Part I — Simple Linear Regression

Start with

\[
\text{charges}_i=\beta_0+\beta_1\text{age}_i+\epsilon_i.
\]

OLS chooses the line minimizing

\[
SSE=\sum_i (y_i-\hat y_i)^2.
\]

In [9]:
simple_model = smf.ols("charges ~ age", data=df).fit()
print(simple_model.summary())

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.089
Method:                 Least Squares   F-statistic:                     131.2
Date:                Sat, 05 Sep 2026   Prob (F-statistic):           4.89e-29
Time:                        11:54:11   Log-Likelihood:                -14415.
No. Observations:                1338   AIC:                         2.883e+04
Df Residuals:                    1336   BIC:                         2.884e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3165.8850    937.149      3.378      0.0

### Reading the age coefficient

The individual hypothesis test is

\[
H_0:\beta_{\text{age}}=0
\qquad\text{vs}\qquad
H_1:\beta_{\text{age}}\neq0.
\]

A small p-value is evidence against a zero conditional slope. It does **not** prove causality.

In [10]:
beta_age = simple_model.params["age"]
ci_age = simple_model.conf_int().loc["age"]

print(f"Estimated age slope: {beta_age:,.2f}")
print(f"95% CI: [{ci_age.iloc[0]:,.2f}, {ci_age.iloc[1]:,.2f}]")

Estimated age slope: 257.72
95% CI: [213.58, 301.87]


In [11]:
age_grid = pd.DataFrame({
    "age": np.linspace(df["age"].min(), df["age"].max(), 150)
})

pred = simple_model.get_prediction(age_grid).summary_frame(alpha=0.05)
plot_df = age_grid.copy()
plot_df["mean"] = pred["mean"].to_numpy()
plot_df["mean_ci_lower"] = pred["mean_ci_lower"].to_numpy()
plot_df["mean_ci_upper"] = pred["mean_ci_upper"].to_numpy()

p = figure(
    width=860, height=450,
    title="Simple OLS: Charges ~ Age",
    x_axis_label="Age",
    y_axis_label="Charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(df["age"], df["charges"], size=6, alpha=0.28, legend_label="observations")
p.line(plot_df["age"], plot_df["mean"], line_width=3, legend_label="estimated mean")
p.varea(
    x=plot_df["age"],
    y1=plot_df["mean_ci_lower"],
    y2=plot_df["mean_ci_upper"],
    alpha=0.20,
    legend_label="95% CI for mean"
)
p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

## Verify the sum-of-squares identity

\[
SST=SSR+SSE
\]

and

\[
R^2=1-\frac{SSE}{SST}.
\]

In [12]:
y = df["charges"].to_numpy()
y_hat = simple_model.fittedvalues.to_numpy()
resid = simple_model.resid.to_numpy()

sst = np.sum((y - y.mean()) ** 2)
ssr = np.sum((y_hat - y.mean()) ** 2)
sse = np.sum(resid ** 2)

display(pd.DataFrame({
    "quantity": ["SST", "SSR", "SSE", "SSR+SSE", "R2 manual", "R2 statsmodels"],
    "value": [sst, ssr, sse, ssr + sse, ssr / sst, simple_model.rsquared]
}))

,quantity,value
0,SST,"196,074,221,568.3671"
1,SSR,"17,530,192,183.1516"
2,SSE,"178,544,029,385.2155"
3,SSR+SSE,"196,074,221,568.3672"
4,R2 manual,0.0894
5,R2 statsmodels,0.0894


# Part II — Multiple Linear Regression

Now fit

\[
charges
=
\beta_0+\beta_1age+\beta_2bmi+\beta_3children
+\text{categorical effects}+\epsilon.
\]

`C(variable)` tells `statsmodels` to construct indicator variables.

In [13]:
multiple_model = smf.ols(
    "charges ~ age + bmi + children + C(sex) + C(smoker) + C(region)",
    data=df
).fit()

print(multiple_model.summary())

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.8
Date:                Sat, 05 Sep 2026   Prob (F-statistic):               0.00
Time:                        11:54:11   Log-Likelihood:                -13548.
No. Observations:                1338   AIC:                         2.711e+04
Df Residuals:                    1329   BIC:                         2.716e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept              -1.19

## Conditional interpretation

In multiple regression a coefficient describes a change in expected response **holding the remaining regressors fixed**.

This is one of the biggest conceptual changes from simple regression.

In [14]:
ci = multiple_model.conf_int()

coef_table = pd.DataFrame({
    "term": multiple_model.params.index,
    "estimate": multiple_model.params.values,
    "std_error": multiple_model.bse.values,
    "p_value": multiple_model.pvalues.values,
    "ci_low": ci.iloc[:, 0].values,
    "ci_high": ci.iloc[:, 1].values,
})

display(coef_table)

,term,estimate,std_error,p_value,ci_low,ci_high
0,Intercept,"-11,938.5386",987.8192,0.0000,"-13,876.3934","-10,000.6837"
1,C(sex)[T.male],-131.3144,332.9454,0.6933,-784.4703,521.8416
2,C(smoker)[T.yes],"23,848.5345",413.1534,0.0000,"23,038.0307","24,659.0384"
3,C(region)[T.northwest],-352.9639,476.2758,0.4588,"-1,287.2982",581.3704
4,C(region)[T.southeast],"-1,035.0220",478.6922,0.0308,"-1,974.0968",-95.9473
5,C(region)[T.southwest],-960.0510,477.9330,0.0448,"-1,897.6364",-22.4656
6,age,256.8564,11.8988,0.0000,233.5138,280.1989
7,bmi,339.1935,28.5995,0.0000,283.0884,395.2985
8,children,475.5005,137.8041,0.0006,205.1633,745.8378


In [15]:
plot_coefs = coef_table[coef_table["term"] != "Intercept"].copy()
source_coef = ColumnDataSource(plot_coefs)

p = figure(
    y_range=list(plot_coefs["term"])[::-1],
    width=920, height=470,
    title="Multiple-Regression Coefficients with 95% CIs",
    x_axis_label="Coefficient estimate",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.segment(
    x0="ci_low", y0="term",
    x1="ci_high", y1="term",
    source=source_coef,
    line_width=2
)
p.scatter(x="estimate", y="term", source=source_coef, size=9)
p.add_layout(Span(location=0, dimension="height", line_dash="dashed"))
p.add_tools(HoverTool(tooltips=[
    ("term", "@term"),
    ("estimate", "@estimate{0,0.00}"),
    ("p-value", "@p_value{0.0000}"),
    ("CI low", "@ci_low{0,0.00}"),
    ("CI high", "@ci_high{0,0.00}")
]))

show(p)

# Part III — Matrix Form

The unifying representation is

\[
\mathbf y=X\beta+\epsilon.
\]

The theoretical OLS solution is

\[
\hat\beta=(X^TX)^{-1}X^Ty.
\]

In actual numerical work, QR/SVD/pseudoinverse approaches are often more stable than explicitly forming a matrix inverse.

In [16]:
X = multiple_model.model.exog
y_model = multiple_model.model.endog
column_names = multiple_model.model.exog_names

display(pd.DataFrame(X, columns=column_names).head())

print("Design matrix shape:", X.shape)
print("Matrix rank:", np.linalg.matrix_rank(X))

,Intercept,C(sex)[T.male],C(smoker)[T.yes],C(region)[T.northwest],C(region)[T.southeast],C(region)[T.southwest],age,bmi,children
0,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,19.0000,27.9000,0.0000
1,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,18.0000,33.7700,1.0000
2,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,28.0000,33.0000,3.0000
3,1.0000,1.0000,0.0000,1.0000,0.0000,0.0000,33.0000,22.7050,0.0000
4,1.0000,1.0000,0.0000,1.0000,0.0000,0.0000,32.0000,28.8800,0.0000


Design matrix shape: (1338, 9)
Matrix rank: 9


In [17]:
beta_manual = np.linalg.pinv(X) @ y_model

comparison = pd.DataFrame({
    "term": column_names,
    "statsmodels": multiple_model.params.to_numpy(),
    "manual_pinv": beta_manual,
})
comparison["abs_difference"] = np.abs(
    comparison["statsmodels"] - comparison["manual_pinv"]
)

display(comparison)

,term,statsmodels,manual_pinv,abs_difference
0,Intercept,"-11,938.5386","-11,938.5386",0.0000
1,C(sex)[T.male],-131.3144,-131.3144,0.0000
2,C(smoker)[T.yes],"23,848.5345","23,848.5345",0.0000
3,C(region)[T.northwest],-352.9639,-352.9639,0.0000
4,C(region)[T.southeast],"-1,035.0220","-1,035.0220",0.0000
5,C(region)[T.southwest],-960.0510,-960.0510,0.0000
6,age,256.8564,256.8564,0.0000
7,bmi,339.1935,339.1935,0.0000
8,children,475.5005,475.5005,0.0000


# Part IV — Individual and Joint Hypothesis Tests

A \(t\)-test handles an individual coefficient.

A partial \(F\)-test can ask whether several restrictions hold simultaneously.

Example:

> after age, BMI, children, sex and region are already in the model, does smoking status improve fit?

In [18]:
reduced_model = smf.ols(
    "charges ~ age + bmi + children + C(sex) + C(region)",
    data=df
).fit()

display(anova_lm(reduced_model, multiple_model))

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,"1,330.0000","171,286,306,931.5965",0.0000,NaN,NaN,NaN
1,"1,329.0000","48,839,532,843.9219",1.0000,"122,446,774,087.6746","3,331.9680",0.0000


In [19]:
print("Overall F-statistic:", multiple_model.fvalue)
print("Overall F-test p-value:", multiple_model.f_pvalue)
print("R^2:", multiple_model.rsquared)
print("Adjusted R^2:", multiple_model.rsquared_adj)

Overall F-statistic: 500.8107416283867
Overall F-test p-value: 0.0
R^2: 0.7509130345985205
Adjusted R^2: 0.7494136397729285


# Part V — Confidence Interval vs Prediction Interval

A confidence interval for

\[
E[Y|X=x_0]
\]

describes uncertainty in the **mean response**.

A prediction interval for

\[
Y_{\text{new}}|X=x_0
\]

also includes person-level random variation, so it is wider.

In [20]:
new_people = pd.DataFrame({
    "age": [25, 40, 55],
    "bmi": [22.0, 30.0, 35.0],
    "children": [0, 2, 1],
    "sex": ["female", "male", "female"],
    "smoker": ["no", "no", "yes"],
    "region": ["northeast", "northwest", "southeast"],
})

prediction_table = multiple_model.get_prediction(new_people).summary_frame(alpha=0.05)
display(pd.concat([new_people, prediction_table], axis=1))

,age,bmi,children,sex,smoker,region,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,25,22.0000,0,female,no,northeast,"1,945.1262",475.0724,"1,013.1526","2,877.0998","-9,983.6689","13,873.9213"
1,40,30.0000,2,male,no,northwest,"8,978.2420",402.4070,"8,188.8197","9,767.6642","-2,940.2632","20,896.7471"
2,55,35.0000,1,female,yes,southeast,"37,349.3447",523.2248,"36,322.9081","38,375.7814","25,412.7979","49,285.8915"


In [21]:
plot_pred = pd.concat([new_people, prediction_table], axis=1).copy()
plot_pred["profile"] = [f"Profile {i+1}" for i in range(len(plot_pred))]

p = figure(
    x_range=plot_pred["profile"].tolist(),
    width=860, height=430,
    title="Mean CIs vs Individual Prediction Intervals",
    y_axis_label="Charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.segment(
    x0=plot_pred["profile"], y0=plot_pred["obs_ci_lower"],
    x1=plot_pred["profile"], y1=plot_pred["obs_ci_upper"],
    line_width=5, alpha=0.35,
    legend_label="95% prediction interval"
)

p.segment(
    x0=plot_pred["profile"], y0=plot_pred["mean_ci_lower"],
    x1=plot_pred["profile"], y1=plot_pred["mean_ci_upper"],
    line_width=9, alpha=0.80,
    legend_label="95% mean CI"
)

p.scatter(
    plot_pred["profile"], plot_pred["mean"],
    size=10, legend_label="predicted mean"
)

p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

# Part VI — ANOVA as Regression

A one-way ANOVA for region is equivalent to a regression model with dummy variables:

\[
Y=\beta_0+\beta_1D_1+\beta_2D_2+\beta_3D_3+\epsilon.
\]

In [22]:
region_model = smf.ols("charges ~ C(region)", data=df).fit()
display(anova_lm(region_model, typ=2))

,sum_sq,df,F,PR(>F)
C(region),"1,300,759,681.3101",3.0000,2.9696,0.0309
Residual,"194,773,461,887.0570","1,334.0000",NaN,NaN


In [23]:
def bokeh_group_summary(data, category, value, title):
    summary = (
        data.groupby(category)[value]
        .agg(["mean", "median", "count"])
        .reset_index()
    )

    p = figure(
        x_range=summary[category].astype(str).tolist(),
        width=860, height=430,
        title=title,
        y_axis_label=f"Mean {value}",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    p.vbar(
        x=summary[category].astype(str),
        top=summary["mean"],
        width=0.65,
        alpha=0.65
    )
    return p, summary

p, region_summary = bokeh_group_summary(
    df, "region", "charges", "Mean Charges by Region"
)
p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

display(region_summary)

,region,mean,median,count
0,northeast,"13,406.3845","10,057.6520",324
1,northwest,"12,417.5754","8,965.7957",325
2,southeast,"14,735.4114","9,294.1319",364
3,southwest,"12,346.9374","8,798.5930",325


## Two-way ANOVA and interaction

Now use smoker and region:

\[
Y=\mu+\alpha_{\text{smoker}}
+\beta_{\text{region}}
+(\alpha\beta)_{\text{smoker,region}}+\epsilon.
\]

The interaction asks whether region differences depend on smoking status.

In [24]:
two_way_model = smf.ols(
    "charges ~ C(smoker) * C(region)",
    data=df
).fit()

display(anova_lm(two_way_model, typ=2))

,sum_sq,df,F,PR(>F)
C(smoker),"120,326,667,100.3616",1.0000,"2,191.3373",0.0000
C(region),"107,523,160.0039",3.0000,0.6527,0.5813
C(smoker):C(region),"1,416,291,755.9258",3.0000,8.5976,0.0000
Residual,"73,030,503,030.7703","1,330.0000",NaN,NaN


In [25]:
group_means = (
    df.groupby(["region", "smoker"], as_index=False)["charges"]
      .mean()
)

p = figure(
    x_range=sorted(df["region"].unique()),
    width=860, height=430,
    title="Mean Charges by Region and Smoking Status",
    x_axis_label="Region",
    y_axis_label="Mean charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

for smoker in ["no", "yes"]:
    tmp = group_means[group_means["smoker"] == smoker]
    p.line(
        tmp["region"], tmp["charges"],
        line_width=3,
        legend_label=f"smoker={smoker}"
    )
    p.scatter(
        tmp["region"], tmp["charges"],
        size=9,
        legend_label=f"smoker={smoker}"
    )

p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

# Part VII — ANCOVA

ANCOVA combines a continuous covariate with a categorical grouping variable.

\[
charges=\beta_0+\beta_1age+\beta_2smoker+\epsilon.
\]

Smoking-group differences are now adjusted for age.

In [26]:
ancova_additive = smf.ols(
    "charges ~ age + C(smoker)",
    data=df
).fit()

display(anova_lm(ancova_additive, typ=2))

,sum_sq,df,F,PR(>F)
C(smoker),"123,917,913,224.8925",1.0000,"3,028.4125",0.0000
age,"19,928,201,786.3771",1.0000,487.0225,0.0000
Residual,"54,626,116,160.3227","1,335.0000",NaN,NaN


## Equal-slopes assumption

The additive ANCOVA assumes the age slope is common across smoking groups.

Test that by adding

\[
age\times smoker.
\]

In [27]:
ancova_interaction = smf.ols(
    "charges ~ age * C(smoker)",
    data=df
).fit()

display(anova_lm(ancova_additive, ancova_interaction))

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,"1,335.0000","54,626,116,160.3227",0.0000,NaN,NaN,NaN
1,"1,334.0000","54,565,065,871.5671",1.0000,"61,050,288.7556",1.4925,0.2220


In [28]:
age_grid = np.linspace(df["age"].min(), df["age"].max(), 100)

p = figure(
    width=860, height=450,
    title="ANCOVA: Age × Smoking Status",
    x_axis_label="Age",
    y_axis_label="Charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

for smoker in ["no", "yes"]:
    sample = df[df["smoker"] == smoker]
    p.scatter(
        sample["age"], sample["charges"],
        size=5, alpha=0.18,
        legend_label=f"{smoker}: observations"
    )

    grid = pd.DataFrame({"age": age_grid, "smoker": smoker})
    yhat = ancova_interaction.predict(grid)

    p.line(
        age_grid, yhat,
        line_width=3,
        legend_label=f"{smoker}: fitted line"
    )

p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

# Part VIII — BMI × Smoking Interaction

The exploratory plot suggests BMI may relate to charges differently for smokers.

Fit

\[
charges=
\beta_0+\beta_1bmi+\beta_2smoker+\beta_3(bmi\times smoker)+\cdots+\epsilon.
\]

In [29]:
interaction_model = smf.ols(
    "charges ~ age + bmi * C(smoker) + children + C(sex) + C(region)",
    data=df
).fit()

print(interaction_model.summary())

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.840
Method:                 Least Squares   F-statistic:                     780.0
Date:                Sat, 05 Sep 2026   Prob (F-statistic):               0.00
Time:                        11:54:12   Log-Likelihood:                -13248.
No. Observations:                1338   AIC:                         2.652e+04
Df Residuals:                    1328   BIC:                         2.657e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept              -2223

In [30]:
display(anova_lm(multiple_model, interaction_model))

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,"1,329.0000","48,839,532,843.9219",0.0000,NaN,NaN,NaN
1,"1,328.0000","31,191,880,482.0135",1.0000,"17,647,652,361.9084",751.3520,0.0000


In [31]:
bmi_grid = np.linspace(df["bmi"].quantile(0.02), df["bmi"].quantile(0.98), 120)

reference = {
    "age": float(df["age"].median()),
    "children": int(df["children"].median()),
    "sex": df["sex"].mode()[0],
    "region": df["region"].mode()[0],
}

p = figure(
    width=860, height=450,
    title="Adjusted BMI Effect by Smoking Status",
    x_axis_label="BMI",
    y_axis_label="Predicted charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

for smoker in ["no", "yes"]:
    grid = pd.DataFrame({
        "age": reference["age"],
        "bmi": bmi_grid,
        "children": reference["children"],
        "sex": reference["sex"],
        "smoker": smoker,
        "region": reference["region"],
    })

    p.line(
        bmi_grid,
        interaction_model.predict(grid),
        line_width=3,
        legend_label=f"smoker={smoker}"
    )

p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

## Interpret the interaction correctly

If non-smoker is the reference group:

\[
\text{BMI slope}_{non-smoker}=\beta_{BMI},
\]

while

\[
\text{BMI slope}_{smoker}
=\beta_{BMI}+\beta_{BMI\times smoker}.
\]

The interaction coefficient is a **difference in slopes**.

In [32]:
params = interaction_model.params

interaction_term = next(
    term for term in params.index
    if "bmi:C(smoker)" in term or "C(smoker)[T.yes]:bmi" in term
)

effects = pd.DataFrame({
    "group": ["non-smoker", "smoker"],
    "estimated_BMI_slope": [
        params["bmi"],
        params["bmi"] + params[interaction_term]
    ]
})

display(effects)

,group,estimated_BMI_slope
0,non-smoker,23.5329
1,smoker,"1,466.6293"


# Part IX — Model Building

Compare candidate formulas using adjusted \(R^2\), AIC, BIC and residual scale.

Do not choose a model only because ordinary \(R^2\) is larger: adding regressors cannot decrease ordinary \(R^2\).

In [33]:
candidate_models = {
    "age_only": simple_model,
    "additive": multiple_model,
    "bmi_smoker_interaction": interaction_model,
    "age_smoker_interaction": smf.ols(
        "charges ~ age * C(smoker) + bmi + children + C(sex) + C(region)",
        data=df
    ).fit(),
    "two_interactions": smf.ols(
        "charges ~ age * C(smoker) + bmi * C(smoker) + children + C(sex) + C(region)",
        data=df
    ).fit(),
}

rows = []
for name, model in candidate_models.items():
    rows.append({
        "model": name,
        "n_params": int(model.df_model + 1),
        "R2": model.rsquared,
        "adj_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "resid_std": np.sqrt(model.mse_resid),
    })

model_metrics = pd.DataFrame(rows).sort_values("AIC")
display(model_metrics)

,model,n_params,R2,adj_R2,AIC,BIC,resid_std
2,bmi_smoker_interaction,10,0.8409,0.8398,"26,515.5704","26,567.5597","4,846.4277"
4,two_interactions,11,0.8409,0.8397,"26,517.5603","26,574.7485","4,848.2352"
3,age_smoker_interaction,10,0.7514,0.7497,"27,113.1540","27,165.1433","6,059.0568"
1,additive,9,0.7509,0.7494,"27,113.5058","27,160.2962","6,062.1023"
0,age_only,2,0.0894,0.0887,"28,833.9487","28,844.3466","11,560.3088"


# Part X — Diagnostics

Classical linear regression relies on assumptions about:

- mean structure / linearity,
- conditional variance,
- independence,
- error distribution for exact small-sample inference.

A high \(R^2\) does not establish any of these.

In [34]:
diagnostic_model = interaction_model
influence = diagnostic_model.get_influence()

diagnostics = pd.DataFrame({
    "fitted": diagnostic_model.fittedvalues,
    "residual": diagnostic_model.resid,
    "studentized": influence.resid_studentized_external,
    "leverage": influence.hat_matrix_diag,
    "cooks_d": influence.cooks_distance[0],
})
diagnostics["sqrt_abs_studentized"] = np.sqrt(
    np.abs(diagnostics["studentized"])
)

display(diagnostics.head())

,fitted,residual,studentized,leverage,cooks_d,sqrt_abs_studentized
0,"22,057.5676","-5,172.6436",-1.0727,0.0099,0.0012,1.0357
1,"2,122.5408",-396.9885,-0.0821,0.0056,0.0000,0.2866
2,"5,773.4296","-1,323.9676",-0.2740,0.0063,0.0000,0.5234
3,"5,924.7030","16,059.7676",3.3364,0.0060,0.0067,1.8266
4,"5,806.3981","-1,939.5429",-0.4011,0.0049,0.0001,0.6333


## 10.1 Residuals vs fitted

Look for curvature, funnel shapes, clusters and isolated points.

In [35]:
diag_source = ColumnDataSource(
    diagnostics.reset_index().rename(columns={"index": "row"})
)

p = figure(
    width=860, height=430,
    title="Residuals vs Fitted Values",
    x_axis_label="Fitted charges",
    y_axis_label="Residual",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    x="fitted", y="residual",
    source=diag_source,
    size=6, alpha=0.45
)
p.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
p.add_tools(HoverTool(tooltips=[
    ("row", "@row"),
    ("fitted", "@fitted{0,0.00}"),
    ("residual", "@residual{0,0.00}")
]))
show(p)

## 10.2 Q-Q plot

A Q-Q plot compares ordered studentized residuals with standard-normal quantiles.

In [53]:
studentized = diagnostics["studentized"].dropna().sort_values().to_numpy()
n = len(studentized)

probs = (np.arange(1, n + 1) - 0.5) / n
theoretical = stats.norm.ppf(probs)

q_theory = np.quantile(theoretical, [0.25, 0.75])
q_sample = np.quantile(studentized, [0.25, 0.75])
slope = (q_sample[1] - q_sample[0]) / (q_theory[1] - q_theory[0])
intercept = q_sample[0] - slope * q_theory[0]

xline = np.array([theoretical.min(), theoretical.max()])
yline = intercept + slope * xline

p = figure(
    width=700, height=500,
    title="Normal Q-Q Plot of Studentized Residuals",
    x_axis_label="Theoretical normal quantile",
    y_axis_label="Observed studentized residual",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(theoretical, studentized, size=6, alpha=0.55)
p.line(xline, yline, line_width=2, line_dash="dashed")
show(p)

## 10.3 Scale-location plot

Plotting \(\sqrt{|r_i|}\) against fitted values highlights changing residual spread.

In [54]:
p = figure(
    width=860, height=430,
    title="Scale-Location Plot",
    x_axis_label="Fitted charges",
    y_axis_label="sqrt(|studentized residual|)",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    diagnostics["fitted"],
    diagnostics["sqrt_abs_studentized"],
    size=6, alpha=0.45
)
show(p)

# Part XI — Outliers, Leverage and Influence

These are not synonyms.

- **Outlier:** unusual response conditional on predictors.
- **High leverage:** unusual predictor vector.
- **Influential point:** materially changes the fitted model.

The hat matrix is

\[
H=X(X^TX)^{-1}X^T,
\]

and \(h_{ii}\) is observation \(i\)'s leverage.

In [55]:
p = figure(
    width=860, height=450,
    title="Studentized Residual vs Leverage",
    x_axis_label="Leverage",
    y_axis_label="Externally studentized residual",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    x="leverage", y="studentized",
    source=diag_source,
    size=7, alpha=0.55
)

p.add_layout(Span(location=2, dimension="width", line_dash="dashed"))
p.add_layout(Span(location=-2, dimension="width", line_dash="dashed"))
p.add_tools(HoverTool(tooltips=[
    ("row", "@row"),
    ("leverage", "@leverage{0.0000}"),
    ("studentized", "@studentized{0.00}"),
    ("Cook's D", "@cooks_d{0.0000}")
]))
show(p)

## Cook's distance

A common exploratory flag is \(D_i>4/n\), but this is a screening rule, not an automatic deletion rule.

In [56]:
cook_threshold = 4 / len(df)

cook_df = diagnostics.reset_index().rename(columns={"index": "row"})
cook_source = ColumnDataSource(cook_df)

p = figure(
    width=920, height=430,
    title="Cook's Distance by Observation",
    x_axis_label="Observation index",
    y_axis_label="Cook's distance",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.segment(
    x0="row", y0=0,
    x1="row", y1="cooks_d",
    source=cook_source,
    alpha=0.5
)
p.scatter(
    x="row", y="cooks_d",
    source=cook_source,
    size=4, alpha=0.6
)
p.add_layout(
    Span(
        location=cook_threshold,
        dimension="width",
        line_dash="dashed",
        line_width=2
    )
)
show(p)

print("4/n threshold:", cook_threshold)

4/n threshold: 0.0029895366218236174


In [57]:
flags = diagnostics.copy()
flags["row"] = flags.index
flags = flags[
    (flags["studentized"].abs() > 2)
    | (flags["cooks_d"] > cook_threshold)
].sort_values("cooks_d", ascending=False)

flagged_data = df.loc[flags["row"]].copy()
flagged_data = flagged_data.join(
    flags.set_index("row")[["studentized", "leverage", "cooks_d"]]
)

display(flagged_data.head(15))

,age,sex,bmi,children,smoker,region,charges,studentized,leverage,cooks_d
1047,22,male,52.5800,1,yes,southeast,"44,501.3982",-3.0956,0.0494,0.0494
128,32,female,17.7650,2,yes,northwest,"32,734.1863",4.2919,0.0221,0.0411
1300,45,male,30.3600,0,yes,southeast,"62,592.8731",6.4212,0.0069,0.0280
1012,61,female,33.3300,4,no,southeast,"36,580.2822",4.4021,0.0099,0.0192
860,37,female,47.6000,2,yes,southwest,"46,113.5110",-2.2332,0.0353,0.0182
577,31,female,38.0950,1,yes,northeast,"58,571.0745",3.4736,0.0136,0.0165
219,24,female,23.2100,0,no,southeast,"25,081.7678",4.5147,0.0073,0.0148
516,20,male,35.3100,1,no,southeast,"27,724.2887",5.2317,0.0055,0.0148
1230,52,male,34.4850,3,yes,northwest,"60,021.3990",3.7335,0.0105,0.0146
819,33,female,35.5300,0,yes,northwest,"55,135.4021",3.6581,0.0108,0.0145


> Do not automatically delete flagged rows. Investigate whether they are errors, rare-but-valid cases, or evidence that the model is incomplete.

# Part XII — Heteroscedasticity

The classical model assumes

\[
Var(\epsilon_i|X)=\sigma^2.
\]

The Breusch-Pagan test checks whether residual variance systematically relates to the regressors.

In [58]:
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(
    diagnostic_model.resid,
    diagnostic_model.model.exog
)

display(pd.DataFrame({
    "quantity": ["LM statistic", "LM p-value", "F statistic", "F p-value"],
    "value": [bp_stat, bp_pvalue, bp_fstat, bp_fpvalue]
}))

,quantity,value
0,LM statistic,10.0973
1,LM p-value,0.3427
2,F statistic,1.1220
3,F p-value,0.3435


If heteroscedasticity is present, one option for inference is heteroscedasticity-consistent covariance estimation such as HC3.

In [59]:
robust_model = diagnostic_model.get_robustcov_results(cov_type="HC3")

se_comparison = pd.DataFrame({
    "term": diagnostic_model.params.index,
    "coef": diagnostic_model.params.values,
    "SE_classical": diagnostic_model.bse.values,
    "SE_HC3": robust_model.bse,
    "p_classical": diagnostic_model.pvalues.values,
    "p_HC3": robust_model.pvalues,
})

display(se_comparison)

,term,coef,SE_classical,SE_HC3,p_classical,p_HC3
0,Intercept,"-2,223.4539",865.6114,780.6892,0.0103,0.0045
1,C(smoker)[T.yes],"-20,415.6112","1,648.2771","1,953.1994",0.0000,0.0000
2,C(sex)[T.male],-500.1460,266.5175,267.4868,0.0608,0.0617
3,C(region)[T.northwest],-585.4780,380.8594,405.2424,0.1245,0.1488
4,C(region)[T.southeast],"-1,210.1312",382.7501,402.1648,0.0016,0.0027
5,C(region)[T.southwest],"-1,231.1077",382.2178,371.6068,0.0013,0.0009
6,age,263.6202,9.5159,9.6267,0.0000,0.0000
7,bmi,23.5329,25.6006,22.5875,0.3581,0.2977
8,bmi:C(smoker)[T.yes],"1,443.0964",52.6470,63.2886,0.0000,0.0000
9,children,516.4034,110.1794,103.5279,0.0000,0.0000


Robust standard errors change estimated **uncertainty**, not the OLS point estimates. They do not repair a badly misspecified mean function.

# Part XIII — Multicollinearity

For predictor \(X_j\),

\[
VIF_j=\frac{1}{1-R_j^2}.
\]

High VIF means the coefficient is being estimated with limited unique predictor information.

In [60]:
X_vif = pd.DataFrame(
    diagnostic_model.model.exog,
    columns=diagnostic_model.model.exog_names
)

vif_table = pd.DataFrame({
    "term": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.to_numpy(), i)
        for i in range(X_vif.shape[1])
    ]
})

display(vif_table.sort_values("VIF", ascending=False))

,term,VIF
0,Intercept,42.6834
8,bmi:C(smoker)[T.yes],25.5331
1,C(smoker)[T.yes],25.2030
4,C(region)[T.southeast],1.6527
5,C(region)[T.southwest],1.5304
3,C(region)[T.northwest],1.5196
7,bmi,1.3874
6,age,1.0175
2,C(sex)[T.male],1.0115
9,children,1.0042


Interpret VIF in context. Interactions and sets of dummy variables can naturally induce correlation. The question is whether coefficient estimates become unstable or scientifically unhelpful.

# Part XIV — Response Transformation

The raw response is right-skewed, so try

\[
Y^*=\log(Y).
\]

Now coefficients are naturally interpreted on a multiplicative scale.

In [61]:
log_model = smf.ols(
    "np.log(charges) ~ age + bmi * C(smoker) + children + C(sex) + C(region)",
    data=df
).fit()

print(log_model.summary())

                            OLS Regression Results                            
Dep. Variable:        np.log(charges)   R-squared:                       0.784
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     534.0
Date:                Sat, 05 Sep 2026   Prob (F-statistic):               0.00
Time:                        12:13:40   Log-Likelihood:                -762.05
No. Observations:                1338   AIC:                             1544.
Df Residuals:                    1328   BIC:                             1596.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  7

In [71]:
compare_diag = pd.DataFrame({
    "raw_fitted": diagnostic_model.fittedvalues,
    "raw_resid": diagnostic_model.resid,
    "log_fitted": log_model.fittedvalues,
    "log_resid": log_model.resid,
})

p1 = figure(
    width=560, height=390,
    title="Raw-Charges Model",
    x_axis_label="Fitted",
    y_axis_label="Residual",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)
p1.scatter(compare_diag["raw_fitted"], compare_diag["raw_resid"], size=5, alpha=0.4)
p1.add_layout(Span(location=0, dimension="width", line_dash="dashed"))

p2 = figure(
    width=560, height=390,
    title="Log-Charges Model",
    x_axis_label="Fitted log(charges)",
    y_axis_label="Residual",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)
p2.scatter(compare_diag["log_fitted"], compare_diag["log_resid"], size=5, alpha=0.4)
p2.add_layout(Span(location=0, dimension="width", line_dash="dashed"))

show(row(p1, p2))

In [63]:
bp_log = het_breuschpagan(log_model.resid, log_model.model.exog)

display(pd.DataFrame({
    "model": ["raw charges", "log charges"],
    "R2_on_model_scale": [diagnostic_model.rsquared, log_model.rsquared],
    "Breusch_Pagan_p": [bp_pvalue, bp_log[1]],
}))

,model,R2_on_model_scale,Breusch_Pagan_p
0,raw charges,0.8409,0.3427
1,log charges,0.7835,0.0000


### Transformation caution

Do not choose a transformation only to chase a statistic. Check whether it improves:

- residual structure,
- variance stability,
- scientific interpretation,
- prediction behavior.

# Part XV — Inference vs Prediction

ST3131 focuses on inference, but held-out prediction checks a complementary question.

- p-values / confidence intervals: what does the sample tell us about model parameters?
- test RMSE / MAE: how accurate are predictions on unseen observations?

In [64]:
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE
)

train_model = smf.ols(
    "charges ~ age + bmi * C(smoker) + children + C(sex) + C(region)",
    data=train_df
).fit()

test_pred = train_model.predict(test_df)

rmse = np.sqrt(mean_squared_error(test_df["charges"], test_pred))
mae = mean_absolute_error(test_df["charges"], test_pred)
test_r2 = r2_score(test_df["charges"], test_pred)

display(pd.DataFrame({
    "metric": ["RMSE", "MAE", "Test R^2"],
    "value": [rmse, mae, test_r2]
}))

,metric,value
0,RMSE,"4,697.6340"
1,MAE,"2,785.9450"
2,Test R^2,0.8538


In [70]:
pred_df = pd.DataFrame({
    "actual": test_df["charges"].to_numpy(),
    "predicted": np.asarray(test_pred),
})

pred_source = ColumnDataSource(pred_df)

p = figure(
    width=650, height=600,
    title="Held-Out Predictions vs Actual Charges",
    x_axis_label="Actual charges",
    y_axis_label="Predicted charges",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p.scatter(
    x="actual", y="predicted",
    source=pred_source,
    size=7, alpha=0.50
)

lo = min(pred_df["actual"].min(), pred_df["predicted"].min())
hi = max(pred_df["actual"].max(), pred_df["predicted"].max())
p.line([lo, hi], [lo, hi], line_dash="dashed", line_width=2)

p.xaxis.formatter = NumeralTickFormatter(format="$0,0")
p.yaxis.formatter = NumeralTickFormatter(format="$0,0")
show(p)

# Part XVI — Guided Exercises

Try these before reading the solution cells.

### Exercise 1
Add an `age × smoker` interaction to the BMI-interaction model. Use a partial \(F\)-test to decide whether the expanded model is supported.

### Exercise 2
Construct a 95% prediction interval for a 45-year-old male smoker with BMI 32, two children, in the southeast.

### Exercise 3
Fit `charges ~ age + I(age**2) + bmi + C(smoker)` and test whether the quadratic age term adds evidence of nonlinearity.

### Exercise 4
Find the five largest Cook's distances. Remove those observations **only for sensitivity analysis**, refit the model, and compare BMI/smoking coefficients.

## Exercise 1 — solution

In [66]:
exercise1_full = smf.ols(
    "charges ~ age * C(smoker) + bmi * C(smoker) + children + C(sex) + C(region)",
    data=df
).fit()

display(anova_lm(interaction_model, exercise1_full))

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,"1,328.0000","31,191,880,482.0135",0.0000,NaN,NaN,NaN
1,"1,327.0000","31,191,645,017.4308",1.0000,"235,464.5827",0.0100,0.9203


## Exercise 2 — solution

In [67]:
profile = pd.DataFrame({
    "age": [45],
    "bmi": [32.0],
    "children": [2],
    "sex": ["male"],
    "smoker": ["yes"],
    "region": ["southeast"],
})

display(
    interaction_model
    .get_prediction(profile)
    .summary_frame(alpha=0.05)
)

,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper
0,"35,478.5108",394.2152,"34,705.1583","36,251.8634","25,939.6209","45,017.4008"


## Exercise 3 — solution

In [68]:
age_linear = smf.ols(
    "charges ~ age + bmi + C(smoker)",
    data=df
).fit()

age_quadratic = smf.ols(
    "charges ~ age + I(age**2) + bmi + C(smoker)",
    data=df
).fit()

display(anova_lm(age_linear, age_quadratic))

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,"1,334.0000","49,513,219,514.1790",0.0000,NaN,NaN,NaN
1,"1,333.0000","49,250,257,858.1079",1.0000,"262,961,656.0711",7.1173,0.0077


## Exercise 4 — solution

In [69]:
top5_idx = diagnostics["cooks_d"].nlargest(5).index
sensitivity_df = df.drop(index=top5_idx)

sensitivity_model = smf.ols(
    "charges ~ age + bmi * C(smoker) + children + C(sex) + C(region)",
    data=sensitivity_df
).fit()

terms_of_interest = [
    term for term in interaction_model.params.index
    if ("bmi" in term.lower()) or ("smoker" in term.lower())
]

sensitivity_comparison = pd.DataFrame({
    "term": terms_of_interest,
    "full_data": interaction_model.params[terms_of_interest].values,
    "without_top5_Cook": sensitivity_model.params[terms_of_interest].values,
})

sensitivity_comparison["absolute_change"] = np.abs(
    sensitivity_comparison["full_data"]
    - sensitivity_comparison["without_top5_Cook"]
)

display(sensitivity_comparison)

,term,full_data,without_top5_Cook,absolute_change
0,C(smoker)[T.yes],"-20,415.6112","-22,892.3961","2,476.7849"
1,bmi,23.5329,26.0946,2.5618
2,bmi:C(smoker)[T.yes],"1,443.0964","1,521.8031",78.7067


# Part XVII — ST3131 Concept Map

| ST3131 concept | Case-study implementation |
|---|---|
| Simple regression | `charges ~ age` |
| OLS | statsmodels + manual pseudoinverse |
| SST / SSR / SSE / \(R^2\) | manually verified |
| Multiple regression | additive insurance model |
| Categorical predictors | `C(sex)`, `C(smoker)`, `C(region)` |
| \(t\)-tests | coefficient table |
| \(F\)-tests | nested-model ANOVA |
| Confidence vs prediction intervals | `get_prediction()` |
| One-way ANOVA | `charges ~ C(region)` |
| Two-way ANOVA | `C(smoker) * C(region)` |
| ANCOVA | `age + C(smoker)` |
| Interaction | `bmi * C(smoker)` |
| Model building | candidate model table |
| Diagnostics | residual, Q-Q, scale-location |
| Leverage | hat-matrix diagonal |
| Outliers | studentized residual |
| Influence | Cook's distance |
| Heteroscedasticity | Breusch-Pagan + HC3 |
| Multicollinearity | VIF |
| Transformation | `log(charges)` |
| Predictive check | held-out RMSE / MAE / \(R^2\) |

# Final Tutor Summary

A reliable regression workflow is:

\[
\boxed{
\text{Understand data}
\rightarrow
\text{Specify}
\rightarrow
\text{Estimate}
\rightarrow
\text{Infer}
\rightarrow
\text{Diagnose}
\rightarrow
\text{Revise}
\rightarrow
\text{Interpret}
}
\]

Questions to ask every time:

1. What is the response?
2. Which predictors are continuous/categorical?
3. Is the proposed functional form plausible?
4. What do coefficients mean conditionally?
5. What uncertainty surrounds them?
6. Which predictors or groups of predictors matter?
7. Are residual assumptions reasonable?
8. Are influential points driving the result?
9. Is variance approximately stable?
10. Is multicollinearity destabilizing inference?
11. Would interactions or transformations make more sense?
12. How uncertain are individual predictions?
13. Do conclusions survive sensitivity checks?

The central lesson of ST3131 is that **fitting a regression is the beginning of the analysis, not the end**.

## Optional extensions

Natural next steps after core ST3131:

- weighted least squares,
- bootstrap intervals,
- cross-validation,
- ridge/lasso,
- generalized linear models,
- spline regression,
- quantile regression,
- causal inference,
- Bayesian linear regression.